# VGG networks

In [1]:
import torch
from torch import nn

[vgg paper](https://arxiv.org/pdf/1409.1556.pdf)

In [2]:
vgg_conf = {
    "VGG11": [64, "M", 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],
    "VGG13": [64, 64, "M", 128, 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],
    "VGG16": [64,64,"M",128,128,"M",256,256,256,"M",512,512,512,"M",512,512,512,"M",],
    "VGG19": [64,64,"M",128,128,"M",256,256,256,256,"M",512,512,512,512,"M",512,512,512,512,"M"]}

## Creating the VGG Model

In [3]:
class VGGModel(nn.Module):
    def __init__(self, c_in, nc=1000, model_type='VGG16'):
        super().__init__()
        self.c_in = c_in
        self.conv = self.create_conv(vgg_conf[model_type])
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(7), 
                                nn.Flatten(),
                                nn.Linear(512*7*7, 4096),
                                nn.ReLU(),
                                nn.Dropout(0.5),
                                nn.Linear(4096, nc))
        
    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x
        
    def create_conv(self, conf):
        layers = []
        c_in = self.c_in
        
        for x in conf:
            if isinstance(x, int):
                c_out = x
                layers += [nn.Sequential(nn.Conv2d(c_in, c_out, kernel_size=(3, 3), padding=(1, 1),  stride=(1,1)),
                                     nn.BatchNorm2d(c_out),
                                     nn.ReLU(inplace=True))]
                c_in = x
            elif x == 'M':
                layers += [nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))]
        return nn.Sequential(*layers)

In [4]:
m = VGGModel(3, 1000, 'VGG11')

In [5]:
m(torch.randn(2, 3, 224, 224)).shape

/home/ubuntu/anaconda3/envs/torchdl0/lib/python3.9/site-packages/torch/nn/functional.py:718: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at  /opt/conda/conda-bld/pytorch_1623448222085/work/c10/core/TensorImpl.h:1156.)
  return torch.max_pool2d(input, kernel_size, stride, padding, dilation, ceil_mode)


torch.Size([2, 1000])